<a href="https://colab.research.google.com/github/Sievv/Sievv/blob/main/Copy_of_PS_EC_KP_sequence_Predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# One environment for PS, EC, KP with identical versions
!pip -q install \
  scikit-learn==1.4.2 \
  xgboost==2.0.3 \
  numpy==1.26.4 \
  scipy==1.11.4 \
  pandas==2.2.2 \
  joblib==1.3.2 \
  threadpoolctl==3.5.0 \
  biopython==1.83 \
  lime==0.2.0.1 \
  skops==0.9.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency

In [ ]:
# Install compatible versions of numpy and scipy
!pip install numpy==1.24.4 scipy==1.10.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 14.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.11.4
    Uninstalling scipy-1.11.4:
      Successfully uninstalled scipy-1.11.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
arviz 0.22.0 requires numpy>=1.26.0, but you have numpy 1.24.4 which is incompatible.
arviz 0.22.0 requires scipy>=1.11.0, but you have scipy 1.10.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 1.24.4 which is incompatible.
jax 0.5.3 requi

In [ ]:
# =============== minimal 5-model setup (no processing, no auto-run) ===============
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix, matthews_corrcoef, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve, recall_score
)
import pandas as pd
import numpy as np
import joblib
import os

# 1) Define the models (as requested)
def get_models(random_state=42):
    return {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=random_state),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=random_state),
        'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=random_state),
        'SVM': SVC(probability=True, random_state=random_state),
    }

# Helper functions for cleaning data - copied from cell HB-gn0eA8sFi
def _clean_X(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    if "Unnamed: 0" in X.columns:
        X = X.drop(columns=["Unnamed: 0"])
    # Convert all columns to numeric, coercing errors and filling NaN
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    return X

def _clean_y(y: pd.Series) -> pd.Series:
    if y.ndim != 1:
        y = y.iloc[:, 0]
    if not np.issubdtype(y.dtype, np.number):
        mapping = {"active":1,"amp":1,"positive":1,"1":1,
                   "inactive":0,"non-amp":0,"negative":0,"0":0}
        y = y.astype(str).str.strip().str.lower().map(mapping)
    return y.astype(int)


# 2) Fit only (uses your X_train, y_train as-is; no preprocessing)
def fit_models(X_train, y_train, models=None):
    # Apply cleaning before fitting
    X_train = _clean_X(X_train)
    y_train = _clean_y(y_train)

    if models is None:
        models = get_models()
    fitted = {}
    for name, clf in models.items():
        clf.fit(X_train, y_train)
        fitted[name] = clf
    return fitted

# 3) Optional: predict utilities (call only if you want)
def predict_all(fitted_models, X_test):
    # Apply cleaning to X_test as well
    X_test = _clean_X(X_test)
    y_pred = {}
    y_prob = {}
    for name, clf in fitted_models.items():
        y_pred[name] = clf.predict(X_test)
        if hasattr(clf, "predict_proba"):
            y_prob[name] = clf.predict_proba(X_test)[:, 1]
        else:
            s = clf.decision_function(X_test)
            s = s[:, 1] if getattr(s, "ndim", 1) > 1 else s
            y_prob[name] = 1/(1+np.exp(-s))
    return y_pred, y_prob

# 4) Optional: quick metric summary (call only if you want)
def summarize(y_test, y_pred_dict, y_prob_dict):
    # Apply cleaning to y_test as well
    y_test = _clean_y(y_test)
    rows = {}
    for name in y_pred_dict:
        y_pred = y_pred_dict[name]
        y_prob = y_prob_dict[name]
        cm = confusion_matrix(y_test, y_pred, labels=[0,1])
        tn, fp, fn, tp = cm.ravel()
        acc  = (tp + tn) / max(tp + tn + fp + fn, 1)
        sens = recall_score(y_test, y_pred)
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        mcc  = matthews_corrcoef(y_test, y_pred)
        roc  = roc_auc_score(y_test, y_prob)
        pr   = average_precision_score(y_test, y_prob)
        rows[name] = {
            "Accuracy": acc, "Recall": sens, "Specificity": spec,
            "MCC": mcc, "ROC_AUC": roc, "PR_AUC": pr,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
        }
    return pd.DataFrame(rows).T

# 5) Optional: save fitted models (call only if you want)
def save_models(fitted_models, out_dir, prefix):
    os.makedirs(out_dir, exist_ok=True)
    for name, clf in fitted_models.items():
        fn = f"{prefix}_{name.replace(' ','_').lower()}.pkl"
        joblib.dump(clf, os.path.join(out_dir, fn))
# =======================================================================

In [ ]:
def preview_dataset(X_train, y_train, X_test, y_test, name="Dataset"):
    print(f"===== {name} Dataset Preview =====")
    print(f"{name} X_train shape: {X_train.shape}")
    display(X_train.head())
    print(f"{name} y_train shape: {y_train.shape}")
    display(y_train.head())
    print(f"{name} X_test shape: {X_test.shape}")
    display(X_test.head())
    print(f"{name} y_test shape: {y_test.shape}")
    display(y_test.head())
    print("======================================")

In [ ]:
XP_train = pd.read_csv('/content/drive/MyDrive/PS/PS imbalance/X_train_features.csv')
yP_train = pd.read_csv('/content/drive/MyDrive/PS/PS imbalance/y_train.csv').squeeze()

XP_test  = pd.read_csv('/content/drive/MyDrive/PS/PS imbalance/X_test_features.csv')
yP_test  = pd.read_csv('/content/drive/MyDrive/PS/PS imbalance/y_test.csv').squeeze()

preview_dataset(XP_train, yP_train, XP_test, yP_test, name="PS")


===== PS Dataset Preview =====
PS X_train shape: (380, 32)


,Sequence,Length,Charge,Hydrophobicity,Molecular_Weight,Number_of_Cysteines,Number_of_Disulfide_Bridges,Isoelectric_Point,Helix,Turn,...,M,N,P,Q,R,S,T,V,W,Y
0,GKFLKKAKKFGKAFVKI,17,7,8,1938.4474,0,0,10.500000,8.0,2.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.882353,0.000000,0.000000
1,EAPQEPQS,8,-2,3,884.8873,0,0,4.250000,1.0,2.0,...,0.000000,0.000000,25.000000,25.000000,0.000000,12.500000,0.000000,0.000000,0.000000,0.000000
2,GMGKKKTDPGRGREIQGIFFKEDSHKESNDCSCGG,35,2,6,3800.1782,2,1,7.710714,5.0,8.0,...,2.857143,2.857143,2.857143,2.857143,5.714286,8.571429,2.857143,0.000000,0.000000,0.000000
3,VRLRSFTTTIHKVNSMVAYKIPVND,25,4,11,2890.3624,0,0,9.316667,11.0,1.0,...,4.000000,8.000000,4.000000,0.000000,8.000000,8.000000,12.000000,16.000000,0.000000,4.000000
4,VRFLRLAFRPCGNANPHKWVRHLSHSDAYVIRI,33,8,16,3930.5512,1,0,9.490000,15.0,3.0,...,0.000000,6.060606,6.060606,0.000000,15.151515,6.060606,0.000000,9.090909,3.030303,3.030303


PS y_train shape: (380,)


,Activity
0,1
1,0
2,0
3,0
4,0


PS X_test shape: (95, 32)


,Sequence,Length,Charge,Hydrophobicity,Molecular_Weight,Number_of_Cysteines,Number_of_Disulfide_Bridges,Isoelectric_Point,Helix,Turn,...,M,N,P,Q,R,S,T,V,W,Y
0,FGAVICANVTGSYGAACNKLSTFTQFSFMVYNSEKNQPTEEKVDCI,46,-1,17,5014.5990,3,1,6.878571,18.0,4.0,...,2.173913,8.695652,2.173913,4.347826,0.000000,8.695652,8.695652,8.695652,0.0,4.347826
1,AFAVPVPDHVAGIPCM,16,0,12,1623.9359,1,0,4.950000,9.0,4.0,...,6.250000,0.000000,18.750000,0.000000,0.000000,0.000000,0.000000,18.750000,0.0,0.000000
2,SSVCRDLDVVRRIICSAGLSLLAEERQENLPDEIYHVYSFALR,43,-1,18,4937.5682,2,1,7.476923,19.0,2.0,...,0.000000,2.325581,2.325581,2.325581,11.627907,11.627907,0.000000,9.302326,0.0,4.651163
3,GIFSKLAGKKIKNLLISGLKG,21,5,9,2185.6941,0,0,10.500000,9.0,4.0,...,0.000000,4.761905,0.000000,0.000000,0.000000,9.523810,0.000000,0.000000,0.0,0.000000
4,AAKKAVASVAKK,12,4,7,1171.4333,0,0,10.500000,7.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,8.333333,0.000000,16.666667,0.0,0.000000


PS y_test shape: (95,)


,Activity
0,0
1,0
2,0
3,1
4,0


In [ ]:
XE_train = pd.read_csv('/content/drive/MyDrive/EC/EC imbalance/X_train_model_features.csv')
yE_train = pd.read_csv('/content/drive/MyDrive/EC/EC imbalance/y_train.csv').squeeze()

XE_test  = pd.read_csv('/content/drive/MyDrive/EC/EC imbalance/X_test_EC.csv')
yE_test  = pd.read_csv('/content/drive/MyDrive/EC/EC imbalance/y_test_EC.csv').squeeze()

preview_dataset(XE_train, yE_train, XE_test, yE_test, name="EC")


===== EC Dataset Preview =====
EC X_train shape: (381, 31)


,Length,Charge,Hydrophobicity,Molecular_Weight,Number_of_Cysteines,Number_of_Disulfide_Bridges,Isoelectric_Point,Helix,Turn,Sheet,...,M,N,P,Q,R,S,T,V,W,Y
0,15,5,8,1759.2284,0,0,10.500000,8.0,1.0,2.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.666667,13.333333,6.666667,0.000000
1,29,0,14,3210.7245,1,0,7.743750,10.0,6.0,1.0,...,3.448276,3.448276,13.793103,3.448276,6.896552,0.000000,6.896552,10.344828,0.000000,0.000000
2,40,8,14,4516.4273,3,1,9.416667,14.0,4.0,2.0,...,7.500000,5.000000,5.000000,0.000000,5.000000,2.500000,10.000000,2.500000,0.000000,5.000000
3,17,-2,6,2148.3312,0,0,5.718750,6.0,1.0,1.0,...,0.000000,5.882353,5.882353,5.882353,5.882353,0.000000,0.000000,5.882353,0.000000,5.882353
4,29,2,10,3132.3981,0,0,8.365000,3.0,7.0,0.0,...,0.000000,3.448276,24.137931,3.448276,6.896552,13.793103,10.344828,0.000000,0.000000,0.000000


EC y_train shape: (381,)


,Activity
0,1
1,0
2,0
3,0
4,0


EC X_test shape: (96, 31)


,Length,Charge,Hydrophobicity,Molecular_Weight,Number_of_Cysteines,Number_of_Disulfide_Bridges,Isoelectric_Point,Helix,Turn,Sheet,...,M,N,P,Q,R,S,T,V,W,Y
0,26,6,0.319231,3323.0,0,0,7.0,0.3,0.2,0.2,...,0.000000,0.000000,0.038462,0.000000,0.000000,0.076923,0.153846,0.076923,0.038462,0.000000
1,42,2,-0.038095,5556.3,0,0,7.0,0.3,0.2,0.2,...,0.000000,0.023810,0.023810,0.071429,0.119048,0.071429,0.071429,0.071429,0.023810,0.047619
2,21,-19,-3.247619,2862.1,0,0,7.0,0.3,0.2,0.2,...,0.000000,0.047619,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,43,1,-0.206977,5710.7,1,0,7.0,0.3,0.2,0.2,...,0.046512,0.000000,0.046512,0.069767,0.069767,0.093023,0.000000,0.069767,0.023256,0.000000
4,17,0,-0.623529,2252.7,0,0,7.0,0.3,0.2,0.2,...,0.058824,0.000000,0.000000,0.117647,0.058824,0.058824,0.000000,0.000000,0.000000,0.000000


EC y_test shape: (96,)


,Activity
0,1
1,0
2,0
3,0
4,0


In [ ]:
XK_train = pd.read_csv('/content/drive/MyDrive/KP/KP imbalance/X_train.csv')
yK_train = pd.read_csv('/content/drive/MyDrive/KP/KP imbalance/y_train.csv').squeeze()

XK_test  = pd.read_csv('/content/drive/MyDrive/KP/KP imbalance/X_test.csv')
yK_test  = pd.read_csv('/content/drive/MyDrive/KP/KP imbalance/y_test.csv').squeeze()

preview_dataset(XK_train, yK_train, XK_test, yK_test, name="KP")


===== KP Dataset Preview =====
KP X_train shape: (408, 31)


,Length,Charge,Hydrophobicity,Molecular_Weight,Number_of_Cysteines,Number_of_Disulfide_Bridges,Isoelectric_Point,Helix,Turn,Sheet,...,M,N,P,Q,R,S,T,V,W,Y
0,24,5,14,2612.2888,2,1,10.500000,13.0,2.0,3.0,...,0.000000,0.000000,4.166667,0.000000,0.000000,4.166667,4.166667,12.500000,0.000000,0.000000
1,10,3,2,1351.5376,0,0,10.450000,3.0,0.0,1.0,...,10.000000,10.000000,0.000000,10.000000,30.000000,0.000000,0.000000,0.000000,0.000000,10.000000
2,25,11,10,2975.6163,1,0,11.181818,9.0,2.0,1.0,...,0.000000,0.000000,4.000000,4.000000,24.000000,0.000000,4.000000,0.000000,0.000000,0.000000
3,29,0,14,3210.7245,1,0,7.743750,10.0,6.0,1.0,...,3.448276,3.448276,13.793103,3.448276,6.896552,0.000000,6.896552,10.344828,0.000000,0.000000
4,31,1,10,3669.9914,1,0,8.914286,11.0,1.0,3.0,...,0.000000,9.677419,3.225806,9.677419,12.903226,12.903226,3.225806,12.903226,3.225806,6.451613


KP y_train shape: (408,)


,Activity
0,1
1,0
2,1
3,0
4,0


KP X_test shape: (103, 31)


,Length,Charge,Hydrophobicity,Molecular_Weight,Number_of_Cysteines,Number_of_Disulfide_Bridges,Isoelectric_Point,Helix,Turn,Sheet,...,M,N,P,Q,R,S,T,V,W,Y
0,29,3,13,3216.7058,0,0,8.950000,12.0,5.0,4.0,...,3.448276,0.000000,10.344828,3.448276,3.448276,6.896552,6.896552,0.000000,0.000000,6.896552
1,40,-3,15,4234.6341,2,1,6.507143,16.0,5.0,3.0,...,0.000000,5.000000,2.500000,5.000000,5.000000,12.500000,2.500000,7.500000,0.000000,5.000000
2,45,-8,22,5523.9594,0,0,4.939286,18.0,8.0,10.0,...,2.222222,0.000000,11.111111,2.222222,2.222222,6.666667,2.222222,2.222222,11.111111,2.222222
3,36,9,18,4390.3795,2,1,10.888889,18.0,2.0,6.0,...,2.777778,2.777778,5.555556,0.000000,11.111111,11.111111,0.000000,2.777778,0.000000,5.555556
4,27,6,16,3095.7296,0,0,11.500000,15.0,3.0,3.0,...,0.000000,7.407407,7.407407,0.000000,11.111111,0.000000,3.703704,11.111111,3.703704,3.703704


KP y_test shape: (103,)


,Activity
0,0
1,0
2,0
3,0
4,1


In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix, matthews_corrcoef, roc_auc_score,
    average_precision_score, recall_score,
    accuracy_score, precision_score, f1_score, roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ==============================
# 1) Define Models
# ==============================
def get_models(random_state=42):
    return {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=random_state, n_jobs=-1),
        'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=random_state),
        'SVM': SVC(probability=True, random_state=random_state),
    }

# ==============================
# 2) Plotting Helpers
# ==============================
def plot_and_save_curves(y_test, y_prob, dataset_name, model_name, save_dir="plots"):
    os.makedirs(save_dir, exist_ok=True)

    if y_prob is not None:
        # ROC Curve
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure()
        plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob):.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {dataset_name} - {model_name}')
        plt.legend(loc="lower right")
        plt.savefig(os.path.join(save_dir, f"{dataset_name}_{model_name}_ROC.png"))
        plt.close()

        # Precision-Recall Curve
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        plt.figure()
        plt.plot(recall, precision, label=f'AP = {average_precision_score(y_test, y_prob):.2f}')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Precision-Recall Curve - {dataset_name} - {model_name}')
        plt.legend(loc="lower left")
        plt.savefig(os.path.join(save_dir, f"{dataset_name}_{model_name}_PR.png"))
        plt.close()

# ==============================
# 3) Training + Evaluation
# ==============================
def evaluate_and_save(X_train, y_train, X_test, y_test, dataset_name, save_dir="models"):
    os.makedirs(save_dir, exist_ok=True)
    results = []

    models = get_models()
    for name, model in models.items():
        print(f"\nTraining {name} on {dataset_name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
        ap = average_precision_score(y_test, y_prob) if y_prob is not None else np.nan
        cm = confusion_matrix(y_test, y_pred)

        # Save model
        model_path = os.path.join(save_dir, f"{dataset_name}_{name.replace(' ', '')}.joblib")
        joblib.dump(model, model_path)

        # Save plots
        plot_and_save_curves(y_test, y_prob, dataset_name, name)

        # Save results
        results.append({
            "Dataset": dataset_name,
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "MCC": mcc,
            "ROC-AUC": roc,
            "Average Precision": ap,
            "Confusion Matrix": cm.tolist()
        })

    return pd.DataFrame(results)


# ==============================
# 4) Load Data (with error handling)
# ==============================
def safe_load_csv(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print(f"⚠️ File not found: {path}")
        return pd.DataFrame()

XK_train = safe_load_csv('/content/drive/MyDrive/KP/KP imbalance/X_train.csv')
yK_train = safe_load_csv('/content/drive/MyDrive/KP/KP imbalance/y_train.csv').squeeze()

XK_test  = safe_load_csv('/content/drive/MyDrive/KP/KP imbalance/X_test.csv')
yK_test  = safe_load_csv('/content/drive/MyDrive/KP/KP imbalance/y_test.csv').squeeze()

XE_train = safe_load_csv('/content/drive/MyDrive/EC/EC imbalance/X_train_features.csv')
yE_train = safe_load_csv('/content/drive/MyDrive/EC/EC imbalance/y_train.csv').squeeze()

XE_test  = safe_load_csv('/content/drive/MyDrive/EC/EC imbalance/X_test_features.csv')
yE_test  = safe_load_csv('/content/drive/MyDrive/EC/EC imbalance/y_test_EC.csv').squeeze()

XP_train = safe_load_csv('/content/drive/MyDrive/PS/PS imbalance/X_train_features.csv')
yP_train = safe_load_csv('/content/drive/MyDrive/PS/PS imbalance/y_train.csv').squeeze()

XP_test  = safe_load_csv('/content/drive/MyDrive/PS/PS imbalance/X_test_features.csv')
yP_test  = safe_load_csv('/content/drive/MyDrive/PS/PS imbalance/y_test.csv').squeeze()

# Drop "Sequence" column if exists
for var in ["XP_train", "XP_test", "XE_train", "XE_test", "XK_train", "XK_test"]:
    if "Sequence" in locals()[var].columns:
        locals()[var] = locals()[var].drop(columns=["Sequence"])



# ==============================
# 6) Run for all datasets
# ==============================
results_P = evaluate_and_save(XP_train, yP_train, XP_test, yP_test, "PeptideP")
results_E = evaluate_and_save(XE_train, yE_train, XE_test, yE_test, "PeptideE")
results_K = evaluate_and_save(XK_train, yK_train, XK_test, yK_test, "PeptideK")

# Combine results
all_results = pd.concat([results_P, results_E, results_K], ignore_index=True)

# Save results
all_results.to_csv("model_results.csv", index=False)
print("\n✅ Training completed. Results saved to model_results.csv and plots/ folder")



Training Random Forest on PeptideP...

Training Gradient Boosting on PeptideP...

Training XGBoost on PeptideP...

Training AdaBoost on PeptideP...


/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



Training SVM on PeptideP...

Training Random Forest on PeptideE...

Training Gradient Boosting on PeptideE...

Training XGBoost on PeptideE...

Training AdaBoost on PeptideE...


/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



Training SVM on PeptideE...

Training Random Forest on PeptideK...

Training Gradient Boosting on PeptideK...

Training XGBoost on PeptideK...

Training AdaBoost on PeptideK...


/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



Training SVM on PeptideK...

✅ Training completed. Results saved to model_results.csv and plots/ folder


In [ ]:
print(results_P)
print(results_E)
print(results_K)

    Dataset              Model  Accuracy  Precision  Recall        F1  \
0  PeptideP      Random Forest  0.957895   0.900000    0.90  0.900000   
1  PeptideP  Gradient Boosting  0.978947   0.909091    1.00  0.952381   
2  PeptideP            XGBoost  0.978947   0.950000    0.95  0.950000   
3  PeptideP           AdaBoost  1.000000   1.000000    1.00  1.000000   
4  PeptideP                SVM  0.789474   0.000000    0.00  0.000000   

        MCC   ROC-AUC  Average Precision    Confusion Matrix  
0  0.873333  0.996000           0.986085  [[73, 2], [2, 18]]  
1  0.940664  1.000000           1.000000  [[73, 2], [0, 20]]  
2  0.936667  0.995333           0.983757  [[74, 1], [1, 19]]  
3  1.000000  1.000000           1.000000  [[75, 0], [0, 20]]  
4  0.000000  0.970667           0.907331  [[75, 0], [20, 0]]  
    Dataset              Model  Accuracy  Precision    Recall        F1  \
0  PeptideE      Random Forest  0.937500   0.941176  0.761905  0.842105   
1  PeptideE  Gradient Boosting  0

In [ ]:
print(results_P)
print(results_E)
print(results_K)

    Dataset              Model  Accuracy  Precision  Recall        F1  \
0  PeptideP      Random Forest  0.957895   0.900000    0.90  0.900000   
1  PeptideP  Gradient Boosting  0.978947   0.909091    1.00  0.952381   
2  PeptideP            XGBoost  0.978947   0.950000    0.95  0.950000   
3  PeptideP           AdaBoost  1.000000   1.000000    1.00  1.000000   
4  PeptideP                SVM  0.789474   0.000000    0.00  0.000000   

        MCC   ROC-AUC  Average Precision    Confusion Matrix  
0  0.873333  0.996000           0.986085  [[73, 2], [2, 18]]  
1  0.940664  1.000000           1.000000  [[73, 2], [0, 20]]  
2  0.936667  0.995333           0.983757  [[74, 1], [1, 19]]  
3  1.000000  1.000000           1.000000  [[75, 0], [0, 20]]  
4  0.000000  0.970667           0.907331  [[75, 0], [20, 0]]  
    Dataset              Model  Accuracy  Precision    Recall        F1  \
0  PeptideE      Random Forest  0.937500   0.941176  0.761905  0.842105   
1  PeptideE  Gradient Boosting  0

In [ ]:
import numpy as np
import joblib
import lime
from lime.lime_tabular import LimeTabularExplainer
from Bio.SeqUtils import molecular_weight
from Bio.Seq import Seq
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# ======================
# 1. Load all models
# ======================
# Example: load your saved models (RandomForest, GradientBoosting, AdaBoost, SVM, XGB)
# Adjust the paths to match where you saved them earlier
models = {
    "PeptideP": {
        "RandomForest": joblib.load("models/PeptideP_RandomForest.joblib"),
        "GradientBoosting": joblib.load("models/PeptideP_GradientBoosting.joblib"),
        "AdaBoost": joblib.load("models/PeptideP_AdaBoost.joblib"),
        "SVM": joblib.load("models/PeptideP_SVM.joblib"),
        "XGBoost": joblib.load("models/PeptideP_XGBoost.joblib"),
    },
    "PeptideE": {
        "RandomForest": joblib.load("models/PeptideE_RandomForest.joblib"),
        "GradientBoosting": joblib.load("models/PeptideE_GradientBoosting.joblib"),
        "AdaBoost": joblib.load("models/PeptideE_AdaBoost.joblib"),
        "SVM": joblib.load("models/PeptideE_SVM.joblib"),
        "XGBoost": joblib.load("models/PeptideE_XGBoost.joblib"),
    },
    "PeptideK": {
        "RandomForest": joblib.load("models/PeptideK_RandomForest.joblib"),
        "GradientBoosting": joblib.load("models/PeptideK_GradientBoosting.joblib"),
        "AdaBoost": joblib.load("models/PeptideK_AdaBoost.joblib"),
        "SVM": joblib.load("models/PeptideK_SVM.joblib"),
        "XGBoost": joblib.load("models/PeptideK_XGBoost.joblib"),
    }
}

# ======================
# 2. Feature Extraction
# ======================
def calculate_charge(sequence):
    positive = sequence.count('K') + sequence.count('R') + sequence.count('H')
    negative = sequence.count('D') + sequence.count('E')
    return positive - negative

def calculate_hydrophobicity(sequence):
    hydrophobic_residues = 'AVILMFYW'
    return sum(sequence.count(aa) for aa in hydrophobic_residues)

def calculate_molecular_weight(sequence):
    return molecular_weight(Seq(sequence), seq_type='protein')

def calculate_number_of_cysteines(sequence):
    return sequence.count('C')

def calculate_number_of_disulfide_bridges(sequence):
    return calculate_number_of_cysteines(sequence) // 2

def calculate_isoelectric_point(sequence):
    pKa_acidic = {'D':3.9, 'E':4.25}
    pKa_basic = {'K':10.5, 'R':12.5, 'H':6.0}
    acidic_count = sum(sequence.count(aa) for aa in pKa_acidic)
    basic_count = sum(sequence.count(aa) for aa in pKa_basic)
    if acidic_count + basic_count == 0:
        return 7.0
    total_charge_pI = sum(pKa_acidic[aa]*sequence.count(aa) for aa in pKa_acidic) + \
                      sum(pKa_basic[aa]*sequence.count(aa) for aa in pKa_basic)
    return total_charge_pI / (acidic_count + basic_count)

def amino_acid_composition(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    total = len(sequence)
    return {aa: (sequence.count(aa) / total * 100 if total > 0 else 0) for aa in amino_acids}

def secondary_structure_features(sequence):
    helix_aa = 'ALIVMFYW'
    sheet_aa = 'FYW'
    turn_aa = 'GP'
    helix = sum(sequence.count(aa) for aa in helix_aa)
    sheet = sum(sequence.count(aa) for aa in sheet_aa)
    turn = sum(sequence.count(aa) for aa in turn_aa)
    flexibility = helix / len(sequence) if len(sequence) > 0 else 0
    return helix, turn, sheet, flexibility

X_train_cols = ['Length', 'Charge', 'Hydrophobicity', 'Molecular_Weight',
                'Number_of_Cysteines', 'Number_of_Disulfide_Bridges', 'Isoelectric_Point',
                'Helix', 'Turn', 'Sheet', 'Flexibility'] + list('ACDEFGHIKLMNPQRSTVWY')

def calc_features_dict(seq):
    features = {
        'Length': len(seq),
        'Charge': calculate_charge(seq),
        'Hydrophobicity': calculate_hydrophobicity(seq),
        'Molecular_Weight': calculate_molecular_weight(seq),
        'Number_of_Cysteines': calculate_number_of_cysteines(seq),
        'Number_of_Disulfide_Bridges': calculate_number_of_disulfide_bridges(seq),
        'Isoelectric_Point': calculate_isoelectric_point(seq),
    }
    features.update(amino_acid_composition(seq))
    helix, turn, sheet, flexibility = secondary_structure_features(seq)
    features.update({'Helix': helix, 'Turn': turn, 'Sheet': sheet, 'Flexibility': flexibility})
    return features

def prepare_features(seq):
    features = calc_features_dict(seq)
    return np.array([features.get(col, 0) for col in X_train_cols])

# ======================
# 3. Prediction Function
# ======================
def classify_peptide(sequence, dataset="PeptideP"):
    feature_vector = prepare_features(sequence).reshape(1, -1)

    model_probs = {}
    for model_name, model in models[dataset].items():
        try:
            prob = model.predict_proba(feature_vector)[:,1][0]
        except:
            # SVM fallback if no predict_proba
            prob = model.decision_function(feature_vector)
            prob = 1 / (1 + np.exp(-prob))  # sigmoid transform
            prob = prob[0]
        model_probs[model_name] = prob

    active = all(prob >= 0.5 for prob in model_probs.values())

    return {
        "Sequence": sequence,
        "Dataset": dataset,
        "Probabilities": model_probs,
        "Active": active
    }

# ======================
# 4. Interactive UI
# ======================
sequence_text = widgets.Text(description="Sequence:", placeholder="Enter peptide sequence")
dataset_dropdown = widgets.Dropdown(options=["PeptideP", "PeptideE", "PeptideK"], description="Dataset")
output_text = widgets.Textarea(layout=widgets.Layout(width="600px", height="300px"), disabled=True)
button = widgets.Button(description="Classify")

def on_button_click(b):
    seq = sequence_text.value.strip().upper()
    dataset = dataset_dropdown.value
    if not seq:
        output_text.value = "⚠️ Please enter a sequence."
        return
    try:
        result = classify_peptide(seq, dataset=dataset)
        output_text.value = f"Dataset: {result['Dataset']}\n" + \
                            f"Sequence: {result['Sequence']}\n" + \
                            "\n".join([f"{m}: {p:.3f}" for m, p in result['Probabilities'].items()]) + \
                            f"\nIs Active: {result['Active']}"
    except Exception as e:
        output_text.value = f"Error: {e}"

button.on_click(on_button_click)
display(sequence_text, dataset_dropdown, button, output_text)


Text(value='', description='Sequence:', placeholder='Enter peptide sequence')

Dropdown(description='Dataset', options=('PeptideP', 'PeptideE', 'PeptideK'), value='PeptideP')

Button(description='Classify', style=ButtonStyle())

Textarea(value='', disabled=True, layout=Layout(height='300px', width='600px'))

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


In [ ]:
# ======================
# 4. Updated Interactive UI (multi-sequence)
# ======================
sequence_input = widgets.Textarea(
    description="Sequences:",
    placeholder="Enter one sequence per line",
    layout=widgets.Layout(width="600px", height="200px")
)

dataset_dropdown = widgets.Dropdown(
    options=["PeptideP", "PeptideE", "PeptideK"],
    description="Dataset"
)

output_text = widgets.Textarea(
    layout=widgets.Layout(width="800px", height="400px"),
    disabled=True
)

button = widgets.Button(description="Classify All")

def on_button_click(b):
    sequences = [s.strip().upper() for s in sequence_input.value.splitlines() if s.strip()]
    dataset = dataset_dropdown.value
    if not sequences:
        output_text.value = "⚠️ Please enter at least one sequence."
        return

    results_str = ""
    for idx, seq in enumerate(sequences, 1):
        try:
            result = classify_peptide(seq, dataset=dataset)
            probs = "\n".join([f"    {m}: {p:.3f}" for m, p in result['Probabilities'].items()])
            result_block = (
                f"🔹 Sequence #{idx}\n"
                f"  Sequence: {seq}\n"
                f"  Dataset: {result['Dataset']}\n"
                f"{probs}\n"
                f"  → Predicted as ACTIVE: {result['Active']}\n"
                f"{'-'*50}\n"
            )
            results_str += result_block
        except Exception as e:
            results_str += f"❌ Error in sequence #{idx} ({seq}): {e}\n{'-'*50}\n"

    output_text.value = results_str

button.on_click(on_button_click)

display(sequence_input, dataset_dropdown, button, output_text)


Textarea(value='', description='Sequences:', layout=Layout(height='200px', width='600px'), placeholder='Enter …

Dropdown(description='Dataset', options=('PeptideP', 'PeptideE', 'PeptideK'), value='PeptideP')

Button(description='Classify All', style=ButtonStyle())

Textarea(value='', disabled=True, layout=Layout(height='400px', width='800px'))

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
